In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "student1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
# Use your actual file name first; if not found, try common alternatives.
logo_candidates = ['Grazioso Salvare Logo.png', 'grazioso_logo.png', 'my-image.png', 'logo.png']
encoded_image = None
for image_filename in logo_candidates:
    if os.path.exists(image_filename):
        encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()
        break

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Center(html.H4('Unique Identifier: Eva - Project Two Dashboard')),
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image) if encoded_image else '',
            style={'height': '100px'}
        )
    ) if encoded_image else html.Center(html.P("Grazioso Salvare Logo (file not found in current folder)")),
    html.Hr(),
    html.Div(

#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        [
            html.Label("Rescue Type Filter"),
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'Reset (Show All)', 'value': 'reset'},
                    {'label': 'Water Rescue', 'value': 'water'},
                    {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                    {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
                ],
                value='reset',
                labelStyle={'display': 'block'}
            )
        ]

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here
                         page_action='native',
                         page_current=0,
                         page_size=10,
                         sort_action='native',
                         filter_action='native',
                         style_table={'overflowX': 'auto'},
                         style_cell={'textAlign': 'left', 'minWidth': '120px', 'width': '120px', 'maxWidth': '250px'},
                         style_header={'fontWeight': 'bold'},
                         row_selectable='single',
                         selected_rows=[0]

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# Helper function for filter queries
def get_filtered_df(filter_type):
    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)
    return dff


@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#
#
#        return (data,columns)
    dff = get_filtered_df(filter_type)
    if dff is None or dff.empty:
        return []
    return dff.to_dict('records')

# Optional callback to keep columns synced if filtered results differ
@app.callback(Output('datatable-id', 'columns'),
              [Input('filter-type', 'value')])
def update_columns(filter_type):
    dff = get_filtered_df(filter_type)
    if dff is None or dff.empty:
        return []
    return [{"name": i, "id": i, "deletable": False, "selectable": True} for i in dff.columns]

# Optional callback to reset selected row when filters change
@app.callback(Output('datatable-id', 'selected_rows'),
              [Input('filter-type', 'value')])
def reset_selected_row(filter_type):
    dff = get_filtered_df(filter_type)
    if dff is None or dff.empty:
        return []
    return [0]

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    if viewData is None or len(viewData) == 0:
        return [
            dcc.Graph(
                figure=px.pie(values=[1], names=['No Data'], title='Preferred Animals')
            )
        ]

    dff = pd.DataFrame.from_dict(viewData)
    if 'breed' not in dff.columns or dff.empty:
        return [
            dcc.Graph(
                figure=px.pie(values=[1], names=['No Breed Data'], title='Preferred Animals')
            )
        ]

    breed_counts = dff['breed'].value_counts().reset_index()
    breed_counts.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure = px.pie(breed_counts, names='breed', values='count', title='Preferred Animals')
        )
    ]

    #return [
    #    dcc.Graph(
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )
    #]

#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]

    dff = pd.DataFrame.from_dict(viewData)

    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Use column names from the AAC dataset
    if "location_lat" not in dff.columns:
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]

    lon_col = None
    for c in ["location_long", "location_lng", "location_lon", "longitude", "lon", "lng"]:
        if c in dff.columns:
            lon_col = c
            break

    if lon_col is None:
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]

    lat = pd.to_numeric(dff.loc[row, "location_lat"], errors="coerce")
    lon = pd.to_numeric(dff.loc[row, lon_col], errors="coerce")

    if pd.isna(lat) or pd.isna(lon):
        return [
            dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
                dl.TileLayer(id="base-layer-id")
            ])
        ]

    breed_val = dff.loc[row, "breed"] if "breed" in dff.columns else "Unknown Breed"
    name_val = dff.loc[row, "name"] if "name" in dff.columns else "Unknown"

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[float(lat), float(lon)], zoom=12, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[float(lat), float(lon)], children=[
                dl.Tooltip(str(breed_val)),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(str(name_val))
                ])
            ])
        ])
    ]



app.run_server(mode='inline', port=8060, debug=True)